# Saved-run evaluation (`best.pt`)
Run from the repository root or `main_notebooks/`. This notebook never retrains or overwrites the selected run. `EVALUATION_DATA_ROOT` can evaluate a checkpoint against a different prepared train/validation split; that override is used consistently for window metrics, continuous metrics, and exported comparison fingerprints. Validation is used for model selection; it is not an independent test set.

**Primary measure:** one alarm per physical double tap on a continuous recording. A correct-side alarm emitted **up to 20 frames before** the labeled second tap is accepted to accommodate auto-labeler timing errors. Late matches are allowed up to 30 frames: the inclusive matching interval is **−20 to +30 frames** (−200 to +300 ms at 100 Hz). Matching is one-to-one; extra or wrong-side alarms still count as errors. Timing is measured at *emission*, not at the probability peak.

Full-recording `forward()` is causal in eval mode and is checked against chunked and single-frame `model.step()` inference. Postprocessing consumes only probabilities available at each emission frame. This assesses the learned model and its streaming implementation under the evaluated policy.

## Fast-copy implementation

This copy preserves the reference grid, populations, metrics, selectors, table
columns, and plots. Pair replay searches precomputed crossing frames and lookahead
peaks instead of scanning quiet frames in Python. Full normal-use sweeps retain
only counts and reuse them across population subsets. Detailed tables are still
built for selected operating points. Input probabilities must not be mutated in
place; rerun the detector and pair-sweep cells after changing upstream inputs.

Pair metrics are cached in memory and (optionally) on disk using actual trace,
truth, exposure, metric-code and timing fingerprints. Budgets/selection modes can
be changed without repeating the sweep. `PAIR_SWEEP_CACHE_DIR=None` disables disk
caching. Exports are **off by default** and use an isolated `fast_evaluation/`
subdirectory. The original notebook and its exports are never overwritten.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.optimize import linear_sum_assignment
from scipy.stats import rankdata
from torch.utils.data import DataLoader

ROOT = Path.cwd().resolve()
if not (ROOT / 'tap_recognition').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'tap_recognition').is_dir(), 'Open from repository root or main_notebooks/'
sys.path.insert(0, str(ROOT))

from tap_recognition.config import TrainConfig
from tap_recognition.model_factory import build_model
from train import build_datasets, frame_soft_ce, window_class_from_probs
from notebooks_analysis.state_reset_analysis import load_recordings, overlap_stream_logits
from notebooks_analysis.recording_types import recording_type_from_filename
from notebooks_analysis.recording_durations import (
    DURATION_BASIS, durations_for_recordings, duration_fingerprint,
)

# Original negative_mining CSVs must be available here (verified against crop provenance).
MINING_SOURCE_DIR = ROOT / 'all_mining_sources'
from notebooks_analysis.hard_negative_mining import MiningModel, predict_recording
from notebooks_analysis.normal_usage_evaluation import load_normal_usage, evaluate_normal_usage
from notebooks_analysis.fast_pair_evaluation import (
    TracePool, normal_usage_counts, PairMetricsCache, sweep_context,
)
from functools import partial
from time import perf_counter
PAIR_SWEEP_CACHE_DIR = ROOT / '.cache/evaluate_fast'  # None disables disk caching.

# Separate full-source population: None = all normal_usage originals; [] = disabled.
# Use a filename list and a held-out folder for an independent normal-use estimate.
NORMAL_USAGE_SOURCE_DIR = MINING_SOURCE_DIR
NORMAL_USAGE_FILES = None
NORMAL_USAGE_CHUNK_FRAMES = 4096

# Change this when evaluating a different run.
# RUN_DIR = ROOT / 'testing_checkpoints/lstm64_features/1310d8(102)feat'
# RUN_DIR = ROOT / 'testing_checkpoints/lstm64_features/d8(102)'
RUN_DIR = ROOT / 'testing_checkpoints/1510d8(52)'

# None evaluates on the checkpoint's recorded split. Set this to a prepared
# dataset root containing train_data/, valid_data/, and session_exclusions.csv
# to compare checkpoints on the same expanded split.
EVALUATION_DATA_ROOT = ROOT / 'data_15_10_26'
# Fix window-negative sampling across checkpoints. This does not change a checkpoint's training seed.
EVALUATION_SEED = 101

# None selects a safe default. With an evaluation-data override, exports go to
# testing_checkpoints/evaluation_exports/ instead of overwriting the checkpoint's
# historical threshold_comparison/ directory.
EVALUATION_OUTPUT_DIR = None

checkpoint = torch.load(RUN_DIR / 'best.pt', map_location='cpu', weights_only=True)
saved = checkpoint['train_config']
cfg = TrainConfig()
for key, value in saved['data'].items():
    setattr(cfg.data, key, value)
for key, value in saved['labels'].items():
    setattr(cfg.labels, key, value)
for key, value in saved['training'].items():
    if key == 'early_stopping':
        for name, setting in value.items():
            setattr(cfg.training.early_stopping, name, setting)
    else:
        setattr(cfg.training, key, value)
checkpoint_training_seed = saved['seed']
cfg.seed = checkpoint_training_seed

if EVALUATION_DATA_ROOT is None:
    evaluation_data_root = None
    for key in ('train_dir', 'val_dir'):
        directories = getattr(cfg.data, key).split(',')
        setattr(cfg.data, key, ','.join(str((ROOT / directory.strip()).resolve()) for directory in directories))
    registry = Path(cfg.data.session_exclusions_file)
    cfg.data.session_exclusions_file = str((ROOT / registry).resolve()) if not registry.is_absolute() else str(registry)
else:
    evaluation_data_root = Path(EVALUATION_DATA_ROOT).resolve()
    for name in ('train_data', 'valid_data'):
        directory = evaluation_data_root / name
        if not directory.is_dir():
            raise FileNotFoundError(f'EVALUATION_DATA_ROOT is missing {directory}')
    registry = evaluation_data_root / 'session_exclusions.csv'
    if not registry.is_file():
        raise FileNotFoundError(f'EVALUATION_DATA_ROOT is missing {registry}')
    cfg.data.train_dir = str(evaluation_data_root / 'train_data')
    cfg.data.val_dir = str(evaluation_data_root / 'valid_data')
    cfg.data.session_exclusions_file = str(registry)

if EVALUATION_SEED is not None:
    cfg.seed = int(EVALUATION_SEED)

# This exact configuration is the single source of truth for every metric and export below.
evaluation_data = cfg.to_dict()['data']
if EVALUATION_OUTPUT_DIR is None:
    EVALUATION_OUTPUT_DIR = (
        RUN_DIR if evaluation_data_root is None
        else ROOT / 'testing_checkpoints/evaluation_exports' / f'{RUN_DIR.name}_on_{evaluation_data_root.name}'
    )
else:
    EVALUATION_OUTPUT_DIR = Path(EVALUATION_OUTPUT_DIR).resolve()
if evaluation_data_root is not None and EVALUATION_OUTPUT_DIR.resolve() == RUN_DIR.resolve():
    raise ValueError('Use a separate EVALUATION_OUTPUT_DIR when overriding evaluation data to preserve historical exports.')
# Keep fast-copy exports separate from the reference notebook.
EVALUATION_OUTPUT_DIR = EVALUATION_OUTPUT_DIR / 'fast_evaluation'
model_cfg = dict(checkpoint['model_config'])
model_cfg['dilations'] = tuple(model_cfg['dilations'])
model = build_model(model_cfg, checkpoint.get('model_type')).cpu().eval()
model.load_state_dict(checkpoint['model_state'])
torch.set_num_threads(2)
run_info = pd.DataFrame([dict(run=str(RUN_DIR), epoch=checkpoint['epoch'],
                             checkpoint_training_seed=checkpoint_training_seed, evaluation_seed=cfg.seed,
                             evaluation_data_root=str(evaluation_data_root) if evaluation_data_root else 'checkpoint split',
                             evaluation_output_dir=str(EVALUATION_OUTPUT_DIR),
                             threshold=cfg.training.prediction_threshold,
                             torch=torch.__version__, numpy=np.__version__, pandas=pd.__version__)])
run_info

In [ ]:
# Evaluation data is configured exclusively by EVALUATION_DATA_ROOT in the setup cell.
# Do not manually change cfg.data paths here: continuous metrics and export fingerprints
# must use the same split as the window diagnostics.

## 1. Training history: underfitting / overfitting
The training notebook must have completed a fresh run to write `history.csv`. Training uses augmentation and dropout; validation does not, so compare **trends** rather than assuming the absolute train loss must be below validation loss. Flat/noisy validation loss as train loss falls suggests a performance plateau (and possibly a generalization gap), not proof of severe overfitting. These plots use window metrics; they cannot establish whether *event-level* latency/false alarms improved at each epoch.

In [ ]:
history_path = RUN_DIR / 'history.csv'
history = pd.read_csv(history_path) if history_path.exists() else pd.DataFrame()
if history.empty:
    print('No history.csv in this run. Re-run the matching training notebook from a fresh kernel and point RUN_DIR at that new run.')
else:
    fig, axes = plt.subplots(1, 4, figsize=(19, 4))
    for ax, metric in zip(axes[:2], ('loss', 'window_acc')):
        for split in ('train', 'val'):
            ax.plot(history.epoch, history[f'{split}_{metric}'], label=split)
        ax.axvline(checkpoint['epoch'], color='gray', linestyle=':', label='best checkpoint')
        ax.set(xlabel='Epoch', ylabel=metric, title=metric)
        ax.legend()
    if {'val_precision', 'val_recall'} <= set(history.columns):
        axes[2].plot(history.epoch, history.val_precision, label='val precision')
        axes[2].plot(history.epoch, history.val_recall, label='val recall')
        if {'val_tp', 'val_fp', 'val_fn'} <= set(history.columns):
            history['val_f1'] = 2*history.val_tp / (2*history.val_tp + history.val_fp + history.val_fn)
            axes[2].plot(history.epoch, history.val_f1, label='val F1')
        axes[2].axvline(checkpoint['epoch'], color='gray', linestyle=':')
        axes[2].set(xlabel='Epoch', ylabel='Score', title='Validation event vs none (windows)')
        axes[2].legend()
    if {'val_fp', 'val_fn'} <= set(history.columns):
        axes[3].plot(history.epoch, history.val_fp, label='val FP')
        axes[3].plot(history.epoch, history.val_fn, label='val FN')
        axes[3].axvline(checkpoint['epoch'], color='gray', linestyle=':')
        axes[3].set(xlabel='Epoch', ylabel='Windows', title='Validation errors (windows)')
        axes[3].legend()
    if 'lr' in history:
        print('Learning-rate schedule is available in history.lr')
    plt.tight_layout()
    plt.show()

In [ ]:
history  # Filter by epoch; compare train_loss / val_loss and train_window_acc / val_window_acc.

## 2. Window-level diagnostics
Each observation is a 300-frame training/validation window. PR-AUC uses one event score per window: the maximum probability of either event class across the window. These are *not* alarm metrics.

In [ ]:
def divide(a, b):
    return float(a / b) if b else float('nan')

def ranking_metrics(truth, score):
    # Average precision = area under the stepwise precision-recall curve.
    truth, score = np.asarray(truth, dtype=bool), np.asarray(score)
    positives, negatives = truth.sum(), (~truth).sum()
    if not positives or not negatives:
        return dict(pr_auc=float('nan'), roc_auc=float('nan'))
    order = np.argsort(-score, kind='stable')
    y, s = truth[order], score[order]
    ends = np.r_[np.flatnonzero(s[:-1] != s[1:]), len(s) - 1]
    tp = np.cumsum(y)[ends]
    precision = tp / (ends + 1)
    pr_auc = np.sum(precision * np.diff(np.r_[0, tp / positives]))
    ranks = rankdata(score, method='average')
    roc_auc = (ranks[truth].sum() - positives * (positives + 1) / 2) / (positives * negatives)
    return dict(pr_auc=float(pr_auc), roc_auc=float(roc_auc))

def binary_metrics(truth, predicted):
    truth, predicted = np.asarray(truth, dtype=bool), np.asarray(predicted, dtype=bool)
    tp, fp = int((truth & predicted).sum()), int((~truth & predicted).sum())
    fn, tn = int((truth & ~predicted).sum()), int((~truth & ~predicted).sum())
    return dict(tp=tp, fp=fp, fn=fn, tn=tn, precision=divide(tp, tp+fp),
                recall=divide(tp, tp+fn), f1=divide(2*tp, 2*tp+fp+fn),
                specificity=divide(tn, tn+fp), accuracy=divide(tp+tn, len(truth)),
                balanced_acc=np.nanmean([divide(tp, tp+fn), divide(tn, tn+fp)]))

LABEL_DELAY_FRAMES = int(checkpoint.get('positive_label_delay_frames', 0))

def shift_event_targets_right(frame_labels, shift):
    """Move event probabilities later and assign remaining mass to none."""
    if shift < 0:
        raise ValueError('shift must be non-negative')
    if shift == 0:
        return frame_labels
    if shift >= frame_labels.shape[1]:
        raise ValueError('shift must be smaller than the number of frames')
    shifted = torch.zeros_like(frame_labels)
    shifted[..., 0] = 1.0
    shifted[:, shift:, 1:] = frame_labels[:, :-shift, 1:]
    shifted[..., 0] = 1.0 - shifted[..., 1:].sum(dim=-1)
    return shifted

@torch.no_grad()
def evaluate_checkpoint(model, loader, device, threshold, event_weight, neg_window_weight):
    """Reproduce validation metrics using the checkpoint's delayed soft targets."""
    model.eval()
    total_loss = correct_frames = total_frames = correct_windows = total_windows = 0
    tp = fp = fn = tn = 0
    for batch in loader:
        imu = batch['imu'].to(device)
        frame_labels = shift_event_targets_right(
            batch['frame_labels'].to(device), LABEL_DELAY_FRAMES
        )
        window_labels = batch['window_label'].to(device)
        logits, _ = model(imu)
        probs = torch.softmax(logits, dim=-1)
        loss = frame_soft_ce(
            logits, frame_labels, event_weight, window_label=window_labels,
            neg_window_weight=neg_window_weight,
        )
        total_loss += loss.item() * imu.size(0)
        pred = probs.argmax(dim=-1)
        true_cls = frame_labels.argmax(dim=-1)
        correct_frames += (pred == true_cls).sum().item()
        total_frames += true_cls.numel()
        window_pred = window_class_from_probs(probs, threshold)
        correct_windows += (window_pred == window_labels).sum().item()
        total_windows += imu.size(0)
        for wp, wl in zip(window_pred > 0, window_labels > 0):
            if wp and wl:
                tp += 1
            elif wp:
                fp += 1
            elif wl:
                fn += 1
            else:
                tn += 1
    n = len(loader.dataset)
    return dict(loss=total_loss / n, frame_acc=correct_frames / total_frames,
                window_acc=correct_windows / total_windows,
                precision=tp / (tp + fp + 1e-8), recall=tp / (tp + fn + 1e-8),
                tp=tp, fp=fp, fn=fn, tn=tn)

_, val_ds = build_datasets(cfg)
val_loader = DataLoader(val_ds, batch_size=cfg.training.batch_size, shuffle=False)
saved_metrics = checkpoint['val_metrics']
measured_metrics = evaluate_checkpoint(model, val_loader, torch.device('cpu'),
                            cfg.training.prediction_threshold, cfg.training.event_loss_weight,
                            cfg.training.neg_window_weight)
metric_keys = ('loss', 'frame_acc', 'window_acc', 'precision', 'recall', 'tp', 'fp', 'fn', 'tn')
metric_reproduction = pd.DataFrame([
    {'metric': key, 'saved': saved_metrics[key], 'current': measured_metrics[key],
     'matches': np.isclose(saved_metrics[key], measured_metrics[key], atol=1e-6)}
    for key in metric_keys
])
if EVALUATION_DATA_ROOT is None and metric_reproduction['matches'].all():
    print('Saved validation metrics independently reproduced.')
elif EVALUATION_DATA_ROOT is not None:
    print('Evaluation-data override is active; saved training-validation metrics are not expected to reproduce.')
else:
    print('Saved validation metrics differ from the current data. The checkpoint does not '
          'store input-label or exclusion-registry contents; compare the saved and current values below.')
print(metric_reproduction.to_string(index=False))
window_rows = []
with torch.inference_mode():
    for index in range(len(val_ds)):
        sample = val_ds[index]
        probs = model(sample['imu'].unsqueeze(0))[0].softmax(-1)
        event = probs[0, :, 1:]
        score = event.max().item()
        predicted = int(window_class_from_probs(probs, cfg.training.prediction_threshold)[0])
        window_rows.append(dict(index=index, file=val_ds.window_meta[index].source_file,
                                true=int(sample['window_label']), pred=predicted,
                                event_score=score, peak_local_frame=int(event.max(-1).values.argmax()),
                                true_global_frame=val_ds.window_meta[index].trigger_frame))
windows = pd.DataFrame(window_rows)
window_summary = pd.DataFrame([binary_metrics(windows.true > 0, windows.pred > 0) |
                               ranking_metrics(windows.true > 0, windows.event_score) |
                               dict(window_3class_accuracy=float((windows.true == windows.pred).mean()))])
window_summary

In [ ]:
window_cm = pd.crosstab(windows.true, windows.pred, rownames=['true'], colnames=['predicted'])
window_cm = window_cm.reindex(index=[0, 1, 2], columns=[0, 1, 2], fill_value=0)
per_class = pd.DataFrame([dict(class_id=c, precision=divide(window_cm.loc[c, c], window_cm[c].sum()),
                               recall=divide(window_cm.loc[c, c], window_cm.loc[c].sum()),
                               f1=divide(2*window_cm.loc[c, c], window_cm[c].sum()+window_cm.loc[c].sum()),
                               support=int(window_cm.loc[c].sum())) for c in (0, 1, 2)])
correct_positive = windows[(windows.true > 0) & (windows.pred > 0)]
print('3-class confusion matrix (rows=true, columns=predicted)')
display(window_cm)
print('Side accuracy given an event was detected:',
      divide((correct_positive.true == correct_positive.pred).sum(), len(correct_positive)))
per_class

In [ ]:
windows  # Filter file, true, pred, event_score and peak_local_frame to inspect individual errors.

## 3. Whole-recording causal frame probabilities
The model state persists within a recording, resets only between recordings, and the high-pass filter is applied once per recording. Frame targets are *hard regions* around each second tap (±10 frames), distinct from the soft labels used for training. Frame accuracy is dominated by background and remains secondary.

In [ ]:
all_recordings, recording_audit, data_manifest = load_recordings(ROOT, evaluation_data, cfg.labels.half_frames)
recordings = [r for r in all_recordings if r.split == 'valid']
assert len(recordings) > 0
duration_audit = durations_for_recordings(ROOT, recordings, MINING_SOURCE_DIR)
duration_lookup = duration_audit.set_index('file')
duration_sha256 = duration_fingerprint(duration_audit)
print('Evaluated input minutes:', duration_audit.evaluated_minutes.sum(),
      '| FP-rate denominator minutes (unique mining sources):', duration_audit.minutes.sum())
display(duration_audit[duration_audit.is_mining])
normal_usage_recordings, normal_usage_manifest, normal_usage_sha256 = load_normal_usage(
    NORMAL_USAGE_SOURCE_DIR, NORMAL_USAGE_FILES)
normal_usage_model = MiningModel(RUN_DIR / 'best.pt', model, cfg.data.highpass, cfg.data.window_samples, '')
normal_usage_probabilities = {rec.path.name: predict_recording(
    rec, normal_usage_model, False, NORMAL_USAGE_CHUNK_FRAMES) for rec in normal_usage_recordings}
print('Full normal_usage originals:', len(normal_usage_recordings),
      '| actual minutes:', normal_usage_manifest.minutes.sum())
display(normal_usage_manifest)
if normal_usage_recordings:
    print('Normal-use rates use every original frame, never crops. This population is separate '
          'from crop diagnostics; do not add their alarm counts/minutes. If its crops were used '
          'for training or threshold tuning, this is not an independent test estimate.')

# Diagnostic-only normal-use population: exact full originals represented by normal_usage
# crops in the validation folder. load_normal_usage deduplicates by content hash, so a
# source contributes its complete duration once even when it produced several crops.
validation_normal_usage_source_files = sorted({
    row.source_file for row in duration_audit.itertuples(index=False)
    if '__crop_' in row.file.lower()
    and recording_type_from_filename(row.source_file) == 'normal_usage'
})
validation_crop_normal_usage_recordings, validation_crop_normal_usage_manifest, validation_crop_normal_usage_sha256 = (
    load_normal_usage(NORMAL_USAGE_SOURCE_DIR, validation_normal_usage_source_files)
)
validation_crop_normal_usage_probabilities = {}
for rec in validation_crop_normal_usage_recordings:
    if rec.path.name in normal_usage_probabilities:
        validation_crop_normal_usage_probabilities[rec.path.name] = normal_usage_probabilities[rec.path.name]
    else:
        validation_crop_normal_usage_probabilities[rec.path.name] = predict_recording(
            rec, normal_usage_model, False, NORMAL_USAGE_CHUNK_FRAMES)
print('Full normal_usage originals referenced by validation crops:',
      len(validation_crop_normal_usage_recordings),
      '| deduplicated actual minutes:', validation_crop_normal_usage_manifest.minutes.sum())
display(validation_crop_normal_usage_manifest)
probabilities, frame_parts = {}, []
with torch.inference_mode():
    for rec in recordings:
        x = torch.from_numpy(rec.imu).unsqueeze(0)
        logits, _ = model(x)
        # Independent causal reference: only new CNN features advance the recurrent state.
        # Full/chunked float32 kernels can accumulate small rounding differences.
        sizes = [min(128, x.shape[1] - i) for i in range(0, x.shape[1], 128)]
        torch.testing.assert_close(overlap_stream_logits(model, x, sizes), logits, atol=1e-4, rtol=1e-4,
                                   msg=lambda message: f'{rec.path.name}: full/chunked inference differs\n{message}')
        p = logits[0].softmax(-1).numpy()
        probabilities[rec.path.name] = p
        target = np.zeros(len(p), dtype=np.int8)
        for frame, _ in rec.events:
            target[max(0, frame-10):min(len(p), frame+11)] = 1
        frame_parts.append(pd.DataFrame(dict(file=rec.path.name, user=rec.user, action=rec.action,
                                             frame=np.arange(len(p)), true=target,
                                             p_none=p[:, 0], p_left=p[:, 1], p_right=p[:, 2],
                                             event_score=p[:, 1:].max(axis=1))))
frames = pd.concat(frame_parts, ignore_index=True)
frame_summary = pd.DataFrame([binary_metrics(frames.true, frames.event_score >= cfg.training.prediction_threshold) |
                              ranking_metrics(frames.true, frames.event_score)])
frame_summary

In [ ]:
frames[(frames['event_score'] > 0.8) & (frames['true'] == 0)]  # Each row is a real recorded frame; filter file and frame range for detailed inspection.

## 4. Streaming alarms and one-to-one event matching
Two *predeclared* policies: `instant` emits on the first crossing (no look-ahead); `lookahead_12f` waits 12 frames like the current default detector. Both require one frame above threshold and enforce a 0.5 s refractory period measured from the selected probability peak. A correct-side alarm is matched one-to-one to a labeled tap when emitted from **25 frames early through 40 frames late**, inclusive. This timing interval accommodates annotation timing errors and applies to all event metrics and threshold selection. Stricter late deadlines are analyzed next. Do not tune a threshold on validation and then call that same validation performance an unbiased test result.

In [ ]:
THRESHOLDS = np.round(np.arange(0.10, 1.01, 0.05), 2)  # includes the training threshold 0.50
POLICIES = {'lookahead_12f': 12}
MAX_LATE_FRAMES = 40
EARLY_ALLOWANCE_FRAMES = 25  # Inclusive matching interval: -25 <= alarm_frame - label_frame <= 40.

def emit_alarms(probs, threshold, look, refractory):
    # All decisions use information available at the emission frame, never future frames.
    alarms, i, n = [], 0, len(probs)
    while i < n:
        score = probs[i, 1:].max()
        if score < threshold:
            i += 1
            continue
        emission = i + look
        if emission >= n:  # An offline end-of-file flush is not an online alarm.
            break
        peak = i + int(probs[i:emission+1, 1:].max(axis=1).argmax())
        cls = int(probs[peak, 1:].argmax() + 1)
        alarms.append(dict(alarm_frame=emission, peak_frame=peak, class_id=cls,
                           probability=float(probs[peak, cls])))
        # Match the detector's lockout: it is measured from the selected peak.
        i = max(emission + 1, peak + max(1, refractory))
    return alarms

def match_alarms(truth, alarms, class_aware=True, early_allowance=None):
    # Min-cost 1:1 assignment: prefer matching as many events as possible, then closest timing.
    if early_allowance is None:
        early_allowance = EARLY_ALLOWANCE_FRAMES
    n, m = len(truth), len(alarms)
    if not n or not m:
        return []
    cost = np.full((n, m+n), 1e6, dtype=float)
    cost[:, m:] = 1e3
    for a, (frame, cls) in enumerate(truth):
        for b, alarm in enumerate(alarms):
            delta = alarm['alarm_frame'] - frame
            if -early_allowance <= delta <= MAX_LATE_FRAMES and (not class_aware or cls == alarm['class_id']):
                cost[a, b] = abs(delta)
    rows, cols = linear_sum_assignment(cost)
    return [(int(a), int(b)) for a, b in zip(rows, cols) if b < m and cost[a, b] < 1e3]

assert not match_alarms([(100, 1)], [dict(alarm_frame=74, class_id=1)])
assert len(match_alarms([(100, 1)], [dict(alarm_frame=75, class_id=1)])) == 1
assert len(match_alarms([(100, 1)], [dict(alarm_frame=98, class_id=1)])) == 1
assert len(match_alarms([(100, 1)], [dict(alarm_frame=140, class_id=1)])) == 1
assert not match_alarms([(100, 1)], [dict(alarm_frame=141, class_id=1)])
assert not match_alarms([(100, 1)], [dict(alarm_frame=75, class_id=2)])
assert not match_alarms([(100, 1)], [dict(alarm_frame=100, class_id=2)])
assert len(match_alarms([(100, 1)], [dict(alarm_frame=98, class_id=1),
                                   dict(alarm_frame=101, class_id=1)])) == 1
assert emit_alarms(np.array([[0., 1., 0.], [0., 1., 0.]]), .5, 0, 5)[0]['alarm_frame'] == 0

session_rows, alarm_rows, truth_rows = [], [], []
for rec in recordings:
    probs = probabilities[rec.path.name]
    duration_info = duration_lookup.loc[rec.path.name]
    minutes = duration_info.minutes
    for policy, look in POLICIES.items():
        for threshold in THRESHOLDS:
            threshold = float(threshold)
            detected = emit_alarms(probs, threshold, look, round(.5 * rec.fs))
            matches = match_alarms(rec.events, detected)
            binary_matches = match_alarms(rec.events, detected, class_aware=False)
            matched_truth = {a: b for a, b in matches}
            matched_alarms = {b: a for a, b in matches}
            session_rows.append(dict(file=rec.path.name, user=rec.user, action=rec.action,
                                     policy=policy, threshold=threshold, minutes=minutes,
                                     evaluated_minutes=duration_info.evaluated_minutes,
                                     source_file=duration_info.source_file,
                                     source_members=duration_info.source_members,
                                     n_true=len(rec.events), n_alarms=len(detected),
                                     tp=len(matches), fp=len(detected)-len(matches),
                                     fn=len(rec.events)-len(matches),
                                     binary_tp=len(binary_matches),
                                     binary_fp=len(detected)-len(binary_matches),
                                     binary_fn=len(rec.events)-len(binary_matches)))
            for a, (frame, cls) in enumerate(rec.events):
                alarm = detected[matched_truth[a]] if a in matched_truth else None
                truth_rows.append(dict(file=rec.path.name, user=rec.user, action=rec.action,
                                       policy=policy, threshold=threshold, true_frame=frame, true_class=cls,
                                       detected=alarm is not None, alarm_frame=alarm['alarm_frame'] if alarm else np.nan,
                                       latency_frames=alarm['alarm_frame']-frame if alarm else np.nan,
                                       fs=rec.fs))
            for b, alarm in enumerate(detected):
                a = matched_alarms.get(b)
                closest = min(rec.events, key=lambda event: abs(alarm['alarm_frame']-event[0])) if rec.events else None
                nearest_delta = alarm['alarm_frame'] - closest[0] if closest else np.nan
                alarm_rows.append(dict(file=rec.path.name, user=rec.user, action=rec.action,
                                       policy=policy, threshold=threshold, **alarm,
                                       matched=a is not None, true_frame=rec.events[a][0] if a is not None else np.nan,
                                       latency_frames=alarm['alarm_frame']-rec.events[a][0] if a is not None else np.nan,
                                       nearest_true_frame=closest[0] if closest else np.nan,
                                       delta_to_nearest_true=nearest_delta,
                                       too_early=bool(nearest_delta < -EARLY_ALLOWANCE_FRAMES) if closest else False))
sessions = pd.DataFrame(session_rows)
alarms = pd.DataFrame(alarm_rows)
truth_events = pd.DataFrame(truth_rows)
print('Validation recordings:', len(recordings), '| positive events:', len([e for r in recordings for e in r.events]))

with pd.option_context('display.max_colwidth', None):
    display(sessions[(sessions['threshold'] == 0.5) & (sessions['policy'] == 'lookahead_12f')])


In [ ]:
event_rows = []
for (policy, threshold), group in sessions.groupby(['policy', 'threshold'], sort=True):
    tp, fp, fn = (int(group[col].sum()) for col in ('tp', 'fp', 'fn'))
    bt, bp, bn = (int(group[col].sum()) for col in ('binary_tp', 'binary_fp', 'binary_fn'))
    negative = group[group.n_true == 0]
    event_rows.append(dict(policy=policy, threshold=threshold, tp=tp, fp=fp, fn=fn,
                           precision=divide(tp, tp+fp), recall=divide(tp, tp+fn),
                           f1=divide(2*tp, 2*tp+fp+fn),
                           binary_f1=divide(2*bt, 2*bt+bp+bn),
                           fp_per_min=divide(fp, group.minutes.sum()),
                           negative_fp_per_min=divide(negative.fp.sum(), negative.minutes.sum())))
event_summary = pd.DataFrame(event_rows)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for policy, group in event_summary.groupby('policy'):
    axes[0].plot(group.recall, group.precision, 'o-', label=policy)
    axes[1].plot(group.threshold, group.negative_fp_per_min, 'o-', label=policy)
axes[0].set(xlabel='Event recall', ylabel='Event precision', title='Class-aware event PR (threshold sweep)')
axes[1].set(xlabel='Probability threshold', ylabel='False alarms/min on negative-only recordings')
for ax in axes: ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
event_summary.iloc[60:]  # Compare threshold, instant vs lookahead, class-aware F1, binary F1 and FP/min.

In [ ]:
sessions  # Filter threshold, policy, user, action or file to see where errors occur.

In [ ]:
alarms[(alarms['true_frame'].isna()) & (alarms['policy'] == 'instant') & (alarms['delta_to_nearest_true'] > -20) & (alarms['threshold'] >= 0.5)].sort_values(by=['delta_to_nearest_true', 'probability']) # Filter too_early == True, threshold, policy, file or matched == False.

In [ ]:
alarms[alarms.too_early]  # Alarms earlier than the configured annotation allowance.

In [ ]:
class_rows = []
for (policy, threshold), positives in truth_events.groupby(['policy', 'threshold']):
    predictions = alarms[(alarms.policy == policy) & (alarms.threshold == threshold)]
    for cls in (1, 2):
        target = positives[positives.true_class == cls]
        classified = predictions[predictions.class_id == cls]
        tp = int(target.detected.sum())
        fp, fn = len(classified)-tp, len(target)-tp
        class_rows.append(dict(policy=policy, threshold=threshold, class_id=cls,
                               support=len(target), tp=tp, fp=fp, fn=fn,
                               precision=divide(tp, tp+fp), recall=divide(tp, tp+fn),
                               f1=divide(2*tp, 2*tp+fp+fn)))
event_per_class = pd.DataFrame(class_rows)
event_per_class  # Per-class detection performance, including wrong-side alarms as FP + FN.

## 5. Latency and recall before a deadline
Latency is `alarm_frame - labeled_second_tap_frame`; negative values are early. The primary event score accepts alarms from **25 frames early through 40 frames late**, inclusive, to accommodate annotation timing errors. **Deadline recall (%) = matched alarms emitted before X frames / all annotated second taps**, including missed taps in the denominator. Strict `<5` therefore accepts matched alarms from −25 through +4 frames. Accepted early alarms also count toward timely recall; their signed latency is retained in the summaries and histograms. At 100 Hz, 25 frames = 250 ms. The histograms include only matched TPs.

In [ ]:
DEADLINES = (5, 10, 15, 20, 30)
latency_rows = []
for (policy, threshold), group in truth_events.groupby(['policy', 'threshold']):
    matched = group[group.detected]
    lat = matched.latency_frames.to_numpy()
    row = dict(policy=policy, threshold=threshold, annotated=len(group), matched=len(matched),
               mean_frames=np.mean(lat) if len(lat) else np.nan,
               median_frames=np.median(lat) if len(lat) else np.nan,
               p90_frames=np.percentile(lat, 90) if len(lat) else np.nan,
               mean_ms=np.mean(lat / matched.fs.to_numpy() * 1000) if len(lat) else np.nan,
               early_matches=int((lat < 0).sum()))
    for deadline in DEADLINES:
        row[f'recall_lt_{deadline}f'] = divide(int((lat < deadline).sum()), len(group))
    latency_rows.append(row)
latency_summary = pd.DataFrame(latency_rows)
# Every latency budget for every threshold/policy, with all missed taps in the denominator.
deadline_rows = []
for (policy, threshold), group in truth_events.groupby(['policy', 'threshold']):
    for budget in range(1, MAX_LATE_FRAMES + 2):
        found = int((group.detected & (group.latency_frames < budget)).sum())
        deadline_rows.append(dict(policy=policy, threshold=threshold, budget_frames=budget,
                                  found=found, annotated=len(group),
                                  percentage=100 * divide(found, len(group))))
deadline_curve = pd.DataFrame(deadline_rows)
for row in latency_summary.itertuples():
    for deadline in DEADLINES:
        curve_row = deadline_curve[(deadline_curve.policy == row.policy) &
                                   (deadline_curve.threshold == row.threshold) &
                                   (deadline_curve.budget_frames == deadline)]
        assert np.isclose(curve_row.percentage.iloc[0] / 100, getattr(row, f'recall_lt_{deadline}f'))
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, policy in zip(axes[:2], POLICIES):
    group = truth_events[(truth_events.policy == policy) &
                         (truth_events.threshold == cfg.training.prediction_threshold)]
    matched_latency = group.loc[group.detected, 'latency_frames']
    ax.hist(matched_latency, bins=np.arange(-EARLY_ALLOWANCE_FRAMES - .5, MAX_LATE_FRAMES + 2.5, 1), color='tab:blue')
    ax.set(xlabel='Emission delay (frames)', ylabel='Matched true taps',
           title=f'{policy} @ {cfg.training.prediction_threshold}: {len(matched_latency)}/{len(group)} TPs')
for policy, group in deadline_curve[deadline_curve.threshold == cfg.training.prediction_threshold].groupby('policy'):
    axes[2].plot(group.budget_frames, group.percentage, label=policy)
for deadline in DEADLINES:
    axes[2].axvline(deadline, color='gray', linestyle=':', alpha=.4)
axes[2].set(xlabel='Alarm emitted before X frames after tap',
            ylabel='Annotated taps detected (%)', ylim=(0, 100),
            xticks=[1, *DEADLINES, MAX_LATE_FRAMES + 1],
            title=f'Recall by latency @ threshold {cfg.training.prediction_threshold}')
axes[2].legend()
plt.tight_layout()
plt.show()

In [ ]:
deadline_curve[(deadline_curve.threshold == cfg.training.prediction_threshold) &
               deadline_curve.budget_frames.isin(DEADLINES)]  # Full filterable table: deadline_curve.

In [ ]:
from matplotlib.ticker import MaxNLocator
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, policy in zip(axes, POLICIES):
    subset = latency_summary[latency_summary.policy == policy]
    for deadline in DEADLINES:
        ax.plot(subset.threshold, 100 * subset[f'recall_lt_{deadline}f'], 'o-', label=f'<{deadline} frames')
    ax.set(title=policy, xlabel='Threshold', ylabel='Annotated taps detected (%)',
           ylim=(0, 100))
    ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
latency_summary  # All DEADLINES and thresholds, by policy; mean latency is conditional on matched events only.

In [ ]:
truth_events  # Includes missed events (detected=False); filter latency_frames and user/action/file.

## 6. Inspect one recording
Set `FILE` and `POLICY` below, then zoom with `START_FRAME` / `END_FRAME`. Red: annotated second tap. Green: actual alarm emission. Orange: probability peak selected for the alarm (may differ from emission). A correct-side alarm up to 20 frames before the annotation can be shown as a matched detection, with one alarm matched per tap.

In [ ]:
# FILE = next(r.path.name for r in recordings if r.events)  # or choose any file in sessions.file
FILE = 'imuStrict_Miguel_knock_twice_left_20260929_153217.csv'
POLICY = 'instant'
THRESHOLD = 0.5
rec = next(r for r in recordings if r.path.name == FILE)
# START_FRAME, END_FRAME = 0, len(rec.imu)  # e.g. 100, 450
START_FRAME, END_FRAME = 0, len(rec.imu)
p = probabilities[FILE]
selected = alarms[(alarms.file == FILE) & (alarms.policy == POLICY) &
                  (alarms.threshold == THRESHOLD)].copy()
x = np.arange(START_FRAME, END_FRAME) / rec.fs
fig, (ax, ap) = plt.subplots(2, 1, figsize=(45, 6), sharex=True)
ax.plot(x, np.linalg.norm(rec.imu[START_FRAME:END_FRAME, :3], axis=1), label='HP acceleration magnitude')
ax.set(ylabel='Magnitude', title=FILE)
ap.plot(x, p[START_FRAME:END_FRAME, 1], label='P(left)', alpha=.8)
ap.plot(x, p[START_FRAME:END_FRAME, 2], label='P(right)', alpha=.8)
ap.axhline(THRESHOLD, color='gray', linestyle=':', label='Threshold')
for frame, _ in rec.events:
    if START_FRAME <= frame < END_FRAME:
        for a in (ax, ap): a.axvline(frame/rec.fs, color='red', alpha=.6, label='Labeled second tap')
for row in selected.itertuples():
    if START_FRAME <= row.alarm_frame < END_FRAME:
        for a in (ax, ap): a.axvline(row.alarm_frame/rec.fs, color='green', alpha=.7, label='Alarm emission')
    if START_FRAME <= row.peak_frame < END_FRAME:
        ap.axvline(row.peak_frame/rec.fs, color='orange', alpha=.5, linestyle=':', label='Selected peak')
ap.set(xlabel='Time (s)', ylabel='Probability', ylim=(-.05, 1.05))
for a in (ax, ap):
    handles, labels = a.get_legend_handles_labels()
    unique = dict(zip(labels, handles))
    a.legend(unique.values(), unique.keys(), loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
selected  # The exact alarms drawn above; filter matched, alarm_frame, peak_frame or latency_frames.

## 7. Choose separate thresholds for left and right
Search pairs jointly: both classes compete for **one** alarm stream and share a 0.5-second cooldown. A frame qualifies for class c only when P(c) reaches that class's threshold. If both qualify, choose the larger probability (ties go to left). During look-ahead, select the largest *qualifying* probability seen so far; the alarm is emitted only when look-ahead ends. Equal thresholds must reproduce the earlier single-threshold detector.

Selection uses class-aware **event** counts, NOT window or frame accuracy. `FP_SELECTION_METRIC` chooses the FP rate used by budget constraints, `recall_target_min_fp`, and FP-rate tie-breakers. The default `negative_fp_per_min` counts **all emitted alarms / exposure minutes only in validation recordings with no annotated double taps**. Mistimed, duplicate, and wrong-side alarms in positive recordings do not enter this rate. The alternative `fp_per_min` counts all unmatched alarms / all validation exposure minutes. Both rates remain in the reports.

For `negative_mining` crops, exposure uses the hash-verified original CSV's actual length, counted **once per unique source represented in validation**, not once per crop. Its minutes are allocated equally across that source's evaluated crops so pooled and per-type sums stay additive. Other files retain their evaluated duration. `duration_audit` / exported `recording_duration_audit.csv` expose `source_minutes`, `source_members`, allocated `minutes`, and actual `evaluated_minutes`. Train replay is accounted separately, never added to validation. Alarms still come from crops: **this is a source-duration-normalized crop score, not a measured full-source or normal-use FP rate**. Source recordings must be available in `MINING_SOURCE_DIR`; missing/mismatched originals raise an error rather than using inferred lengths.

`normal_usage_fp_per_min` is different: **actual alarms on complete normal_usage originals / all minutes of those originals**, including files with no alarms. `NORMAL_USAGE_SOURCE_DIR` and `NORMAL_USAGE_FILES` define that separate population; inference preserves state across source sessions and resets between files. Normal-usage crops remain `hard_negative` crop diagnostics/training examples and never substitute for these full-source traces. Set `FP_SELECTION_METRIC='normal_usage_fp_per_min'` to use this measured rate for budgets, FP minimization, and FP-rate tie-breakers. Recall and F1 still come from the labeled validation population. With no selected normal originals, the rate is undefined and a normal-use FP-budget selection is infeasible. Exports preserve the full-source manifest/hash and per-recording/alarm tables. This is only an independent test population if its recordings/crops were kept out of training and threshold tuning.

`fp_budget_max_recall` requires the selected FP rate strictly below `FP_PER_MIN_LIMIT`, then maximizes mean per-side event recall; `timely_recall_fp_budget` instead maximizes deadline recall. `recall_target_min_fp` requires each side's recall ≥ `RECALL_TARGET`, then minimizes the selected FP rate. `recall_target_max_threshold` requires that same recall and maximizes the mean threshold. `max_f1` still maximizes mean(left F1, right F1): its primary F1 score still penalizes unmatched alarms in positive recordings, regardless of `FP_SELECTION_METRIC`. Only its FP-rate tie-breaker changes. A constraint with no eligible pair (including undefined negative-only rate when there are no negative recordings) is marked infeasible, never silently relaxed.

These are **tuned-validation** scores: the same recordings choose and evaluate thresholds. To claim generalization, freeze the chosen pair and apply it to separate test recordings. Keep the same grid, alarm policy and metric definitions for all models.

In [ ]:
# Edit this cell to choose the operating points you want to compare.
SELECTION_MODES = ['max_f1', 
                   'recall_target_max_threshold', 
                #    'recall_target_min_fp',
                   'fp_budget_max_recall',  
                #    'timely_recall_fp_budget'
               #  'negative_fp_per_min',
                   ]
RECALL_TARGET = 0.8
FP_SELECTION_METRIC = 'negative_fp_per_min'  # Or 'normal_usage_fp_per_min' for measured full-source FP/min, or 'fp_per_min'.
FP_PER_MIN_LIMIT = 0.28  # Strictly less than this many alarms/min for FP_SELECTION_METRIC.
TIMELY_BUDGET_FRAMES = 10  # Strict <10 frames, including matches within EARLY_ALLOWANCE_FRAMES.


PAIR_THRESHOLDS = np.round(np.arange(0.5, 0.951, 0.05), 2)  # Predeclared common grid
SIDE_DIAGNOSTIC_FIXED_OPPOSITE_THRESHOLD = 0.50  # Hold the other side here for 1D diagnostic curves.


valid_modes = {'max_f1', 'recall_target_max_threshold', 'recall_target_min_fp',
               'fp_budget_max_recall', 'timely_recall_fp_budget'}
assert len(SELECTION_MODES) == len(set(SELECTION_MODES)) and set(SELECTION_MODES) <= valid_modes, (
    'Use selector names in SELECTION_MODES; FP_SELECTION_METRIC chooses the FP-rate column.')
assert 0 < RECALL_TARGET <= 1 and FP_PER_MIN_LIMIT > 0 and TIMELY_BUDGET_FRAMES > 0
assert FP_SELECTION_METRIC in {'negative_fp_per_min', 'fp_per_min', 'normal_usage_fp_per_min'}
assert len(PAIR_THRESHOLDS) and np.all((PAIR_THRESHOLDS > 0) & (PAIR_THRESHOLDS < 1))
assert np.isclose(PAIR_THRESHOLDS, SIDE_DIAGNOSTIC_FIXED_OPPOSITE_THRESHOLD).any(), (
    'SIDE_DIAGNOSTIC_FIXED_OPPOSITE_THRESHOLD must be in PAIR_THRESHOLDS')
selection_settings = pd.DataFrame([dict(modes=', '.join(SELECTION_MODES),
                                        recall_target=RECALL_TARGET, fp_per_min_limit=FP_PER_MIN_LIMIT,
                                        fp_selection_metric=FP_SELECTION_METRIC,
                                        timely_budget_frames=TIMELY_BUDGET_FRAMES,
                                        threshold_grid=', '.join(map(str, PAIR_THRESHOLDS)))])
selection_settings

In [ ]:
# Recreate after changing upstream probabilities; prepared traces are immutable snapshots.
fast_traces = TracePool()
emit_alarms_pair = fast_traces.emit
fast_normal_usage_counts = partial(normal_usage_counts, fast_traces)

# At equal thresholds the new detector must reproduce every existing alarm.
for rec in recordings:
    for policy, look in POLICIES.items():
        for threshold in (0.2, 0.5, 0.8):
            original = emit_alarms(probabilities[rec.path.name], threshold, look, round(.5 * rec.fs))
            paired = emit_alarms_pair(probabilities[rec.path.name], threshold, threshold, look, round(.5 * rec.fs))
            assert original == paired, (rec.path.name, policy, threshold)

def choose_classes(scores, left_threshold, right_threshold):
    # [N, 2] per-frame or per-window probabilities; none=0, left=1, right=2.
    eligible = np.where(scores >= [left_threshold, right_threshold], scores, -np.inf)
    return np.where(np.isfinite(eligible).any(axis=1), 1 + eligible.argmax(axis=1), 0)

assert choose_classes(np.array([[.6, .7], [.8, .4], [.1, .1]]), .5, .8).tolist() == [1, 1, 0]
print('Equal-threshold replay and left/right eligibility checks passed.')

In [ ]:
# Cache per-class window scores; full-recording probabilities are already cached above.
window_side_scores = []
with torch.inference_mode():
    for index in range(len(val_ds)):
        p = model(val_ds[index]['imu'].unsqueeze(0))[0].softmax(-1)[0, :, 1:]
        window_side_scores.append(p.max(dim=0).values.numpy())
window_side_scores = np.stack(window_side_scores)
assert np.allclose(window_side_scores.max(axis=1), windows.event_score, atol=1e-6)
window_truth = windows.true.to_numpy()
frame_side_scores = frames[['p_left', 'p_right']].to_numpy()
frame_true_class = np.concatenate([
    np.array([next((cls for frame, cls in rec.events if abs(i - frame) <= 10), 0)
              for i in range(len(rec.imu))], dtype=np.int8) for rec in recordings
])
assert np.array_equal(frame_true_class > 0, frames.true.to_numpy().astype(bool))
side_score_audit = pd.DataFrame([dict(windows=len(window_truth), frames=len(frame_true_class),
                                      left_truth_frames=int((frame_true_class == 1).sum()),
                                      right_truth_frames=int((frame_true_class == 2).sum()))])
side_score_audit

In [ ]:
pair_cache = PairMetricsCache(PAIR_SWEEP_CACHE_DIR, sweep_context(
    recordings, probabilities, duration_audit,
    [(normal_usage_recordings, normal_usage_probabilities),
     (validation_crop_normal_usage_recordings, validation_crop_normal_usage_probabilities)],
    dict(policies=POLICIES, deadlines=DEADLINES, timely_budget_frames=TIMELY_BUDGET_FRAMES,
         early_allowance_frames=EARLY_ALLOWANCE_FRAMES, max_late_frames=MAX_LATE_FRAMES,
         metric_source_sha256='f265e97a67358f5799b04fd05cdfb68c27301ad0590b329f299f9da34cec0e9c'),
))

@pair_cache.memoize
def evaluate_pair_events(left_threshold, right_threshold, policy):
    look = POLICIES[policy]
    tp = fp = fn = binary_tp = binary_fp = binary_fn = 0
    true_by_class = {1: 0, 2: 0}
    predicted_by_class = {1: 0, 2: 0}
    matched_by_class = {1: 0, 2: 0}
    timely_by_class = {1: 0, 2: 0}
    deadlines = {d: 0 for d in DEADLINES}
    deadline_sides = {(cls, d): 0 for cls in (1, 2) for d in DEADLINES}
    minutes = negative_minutes = negative_fp = 0.0
    latency_ms = []
    side_latency_ms = {1: [], 2: []}
    negative_fp_by_class = {1: 0, 2: 0}
    for rec in recordings:
        detected = emit_alarms_pair(probabilities[rec.path.name], left_threshold, right_threshold,
                                    look, round(.5 * rec.fs))
        matches = match_alarms(rec.events, detected)
        binary = match_alarms(rec.events, detected, class_aware=False)
        tp += len(matches)
        fp += len(detected) - len(matches)
        fn += len(rec.events) - len(matches)
        binary_tp += len(binary)
        binary_fp += len(detected) - len(binary)
        binary_fn += len(rec.events) - len(binary)
        duration = duration_lookup.loc[rec.path.name, 'minutes']
        minutes += duration
        if not rec.events:
            negative_minutes += duration
            negative_fp += len(detected)
            for alarm in detected:
                negative_fp_by_class[alarm['class_id']] += 1
        for _, cls in rec.events:
            true_by_class[cls] += 1
        for alarm in detected:
            predicted_by_class[alarm['class_id']] += 1
        for truth_i, alarm_i in matches:
            frame, cls = rec.events[truth_i]
            delta = detected[alarm_i]['alarm_frame'] - frame
            matched_by_class[cls] += 1
            latency_ms.append(delta / rec.fs * 1000)
            side_latency_ms[cls].append(delta / rec.fs * 1000)
            if delta < TIMELY_BUDGET_FRAMES:
                timely_by_class[cls] += 1
            for d in DEADLINES:
                if delta < d:
                    deadlines[d] += 1
                    deadline_sides[cls, d] += 1
    class_rows = {}
    for cls in (1, 2):
        ctp = matched_by_class[cls]
        cfp = predicted_by_class[cls] - ctp  # Wrong-side alarms are FP for the predicted side.
        cfn = true_by_class[cls] - ctp   # Wrong-side alarms are FN for the true side.
        class_rows[cls] = dict(tp=ctp, fp=cfp, fn=cfn, support=true_by_class[cls],
                               precision=divide(ctp, ctp + cfp), recall=divide(ctp, ctp + cfn),
                               f1=divide(2 * ctp, 2 * ctp + cfp + cfn),
                               timely_recall=divide(timely_by_class[cls], true_by_class[cls]),
                               fp_per_min=divide(cfp, minutes),
                               minutes=minutes, negative_minutes=negative_minutes,
                               evaluated_minutes=float(duration_audit.evaluated_minutes.sum()),
                               negative_fp_per_min=divide(negative_fp_by_class[cls], negative_minutes),
                               mean_latency_ms=np.mean(side_latency_ms[cls]) if side_latency_ms[cls] else np.nan,
                               median_latency_ms=np.median(side_latency_ms[cls]) if side_latency_ms[cls] else np.nan,
                               p90_latency_ms=np.percentile(side_latency_ms[cls], 90) if side_latency_ms[cls] else np.nan,
                               **{f'recall_lt_{d}f': divide(deadline_sides[cls, d], true_by_class[cls])
                                  for d in DEADLINES})
    assert sum(class_rows[c]['tp'] for c in (1, 2)) == tp
    assert sum(class_rows[c]['fp'] for c in (1, 2)) == fp
    assert sum(class_rows[c]['fn'] for c in (1, 2)) == fn
    macro_f1 = np.nanmean([class_rows[c]['f1'] for c in (1, 2)])
    macro_recall = np.nanmean([class_rows[c]['recall'] for c in (1, 2)])
    macro_timely_recall = np.nanmean([class_rows[c]['timely_recall'] for c in (1, 2)])
    overall = dict(tp=tp, fp=fp, fn=fn, annotated=tp + fn,
                   precision=divide(tp, tp + fp), recall=divide(tp, tp + fn),
                   f1=divide(2 * tp, 2 * tp + fp + fn),
                   binary_precision=divide(binary_tp, binary_tp + binary_fp),
                   binary_recall=divide(binary_tp, binary_tp + binary_fn),
                   binary_f1=divide(2 * binary_tp, 2 * binary_tp + binary_fp + binary_fn),
                   macro_f1=macro_f1, macro_recall=macro_recall,
                   macro_timely_recall=macro_timely_recall,
                   fp_per_min=divide(fp, minutes),
                   negative_fp_per_min=divide(negative_fp, negative_minutes),
                   minutes=minutes, negative_minutes=negative_minutes,
                   evaluated_minutes=float(duration_audit.evaluated_minutes.sum()),
                   mean_latency_ms=np.mean(latency_ms) if latency_ms else np.nan,
                   median_latency_ms=np.median(latency_ms) if latency_ms else np.nan,
                   p90_latency_ms=np.percentile(latency_ms, 90) if latency_ms else np.nan,
                   **{f'recall_lt_{d}f': divide(deadlines[d], tp + fn) for d in DEADLINES})
    normal_all, normal_sides, _, _ = fast_normal_usage_counts(
        normal_usage_recordings, normal_usage_probabilities, left_threshold, right_threshold, look)
    overall.update(normal_all)
    for cls in (1, 2):
        class_rows[cls].update(normal_sides[cls])

    # This smaller full-source population is only for the side-threshold plots below.
    # It is derived from validation crop provenance, hash-deduplicated, and never added
    # to crop metrics or used by operating-point selection.
    _, validation_crop_normal_sides, _, _ = fast_normal_usage_counts(
        validation_crop_normal_usage_recordings, validation_crop_normal_usage_probabilities,
        left_threshold, right_threshold, look)
    for cls in (1, 2):
        class_rows[cls]['validation_crop_normal_usage_fp_per_min'] = (
            validation_crop_normal_sides[cls]['normal_usage_fp_per_min']
        )
        class_rows[cls]['validation_crop_normal_usage_minutes'] = (
            validation_crop_normal_sides[cls]['normal_usage_minutes']
        )
    return overall, class_rows

sweep_started = perf_counter()
sweep_rows = []
for policy in POLICIES:
    for left_threshold in PAIR_THRESHOLDS:
        for right_threshold in PAIR_THRESHOLDS:
            overall, sides = evaluate_pair_events(float(left_threshold), float(right_threshold), policy)
            sweep_rows.append(dict(policy=policy, left_threshold=float(left_threshold),
                                   right_threshold=float(right_threshold), **overall,
                                   left_recall=sides[1]['recall'], right_recall=sides[2]['recall'],
                                   left_f1=sides[1]['f1'], right_f1=sides[2]['f1'],
                                   left_fp_per_min=sides[1]['fp_per_min'],
                                   right_fp_per_min=sides[2]['fp_per_min'],
                                   left_validation_crop_normal_usage_fp_per_min=(
                                       sides[1]['validation_crop_normal_usage_fp_per_min']),
                                   right_validation_crop_normal_usage_fp_per_min=(
                                       sides[2]['validation_crop_normal_usage_fp_per_min'])))
pair_sweep = pd.DataFrame(sweep_rows)
# Compare only thresholds present in BOTH sweeps: event_summary uses the coarse THRESHOLDS grid.
shared_thresholds = [t for t in THRESHOLDS if np.isclose(PAIR_THRESHOLDS, t).any()]
for policy in POLICIES:
    for threshold in shared_thresholds:
        original = event_summary[(event_summary.policy == policy) &
                                 np.isclose(event_summary.threshold, threshold)].iloc[0]
        paired = pair_sweep[(pair_sweep.policy == policy) &
                            np.isclose(pair_sweep.left_threshold, threshold) &
                            np.isclose(pair_sweep.right_threshold, threshold)].iloc[0]
        assert (paired.tp, paired.fp, paired.fn) == (original.tp, original.fp, original.fn)
        assert np.allclose([paired.fp_per_min, paired.negative_fp_per_min],
                           [original.fp_per_min, original.negative_fp_per_min], equal_nan=True)
print(f'Equal-threshold check: {len(shared_thresholds)} shared grid values per policy.'
      if shared_thresholds else 'No shared grid values; the direct detector replay check above still applies.')
sweep_finished = perf_counter()
print(f'Pair sweep: {sweep_finished - sweep_started:.2f} s | '
      f'computed={pair_cache.misses}, disk hits={pair_cache.disk_hits}, memory hits={pair_cache.hits}')
pair_sweep


### Per-side threshold diagnostics

Each curve sweeps one side while holding the other side at `SIDE_DIAGNOSTIC_FIXED_OPPOSITE_THRESHOLD`; this is necessary because the two classes share one alarm stream and cooldown. The top row uses the current whole-validation `fp_per_min` denominator. The bottom row uses only complete `normal_usage` originals referenced by validation crops, deduplicated by source hash; it is a full-source diagnostic and does not alter threshold selection.


In [ ]:
def plot_side_threshold_curve(axis, curve, threshold_column, recall_column, fp_column, title):
    recall_line, = axis.plot(curve[threshold_column], curve[recall_column] * 100,
                             marker='o', color='tab:blue', label='Recall')
    axis.set(xlabel=f'{threshold_column.removesuffix("_threshold").title()} threshold',
             ylabel='Recall (%)', ylim=(0, 100), title=title)
    fp_axis = axis.twinx()
    fp_line, = fp_axis.plot(curve[threshold_column], curve[fp_column], marker='s',
                            color='tab:red', label='False positives / min')
    fp_axis.set_ylabel('False positives / min', color='tab:red')
    fp_axis.set(ylim=(0, 0.1))
    fp_axis.tick_params(axis='y', labelcolor='tab:red')
    axis.legend([recall_line, fp_line], [recall_line.get_label(), fp_line.get_label()],
                loc='best', fontsize=8)


for policy in POLICIES:
    left_curve = pair_sweep[(pair_sweep.policy == policy) &
                            np.isclose(pair_sweep.right_threshold,
                                       SIDE_DIAGNOSTIC_FIXED_OPPOSITE_THRESHOLD)].sort_values('left_threshold')
    right_curve = pair_sweep[(pair_sweep.policy == policy) &
                             np.isclose(pair_sweep.left_threshold,
                                        SIDE_DIAGNOSTIC_FIXED_OPPOSITE_THRESHOLD)].sort_values('right_threshold')
    assert len(left_curve) == len(PAIR_THRESHOLDS) == len(right_curve)

    fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex='col')
    plot_side_threshold_curve(axes[0, 0], left_curve, 'left_threshold', 'left_recall',
                              'left_fp_per_min',
                              'Left recall and whole-validation FP/min')
    plot_side_threshold_curve(axes[0, 1], right_curve, 'right_threshold', 'right_recall',
                              'right_fp_per_min',
                              'Right recall and whole-validation FP/min')
    plot_side_threshold_curve(axes[1, 0], left_curve, 'left_threshold', 'left_recall',
                              'left_validation_crop_normal_usage_fp_per_min',
                              'Left recall and validation-source normal-use FP/min')
    plot_side_threshold_curve(axes[1, 1], right_curve, 'right_threshold', 'right_recall',
                              'right_validation_crop_normal_usage_fp_per_min',
                              'Right recall and validation-source normal-use FP/min')
    fig.suptitle(
        f'{policy}: opposite-side threshold fixed at {SIDE_DIAGNOSTIC_FIXED_OPPOSITE_THRESHOLD:.2f}; '
        f'normal-use row = {len(validation_crop_normal_usage_recordings)} deduplicated full originals '
        f'({validation_crop_normal_usage_manifest.minutes.sum():.2f} min)',
        y=1.02,
    )
    fig.tight_layout()
    plt.show()

    # A single common threshold makes all three quantities directly comparable:
    # left/right recall comes from the labeled validation set, while FP/min uses
    # every configured full normal_usage original, including training-crop sources.
    equal_threshold_curve = pair_sweep[(pair_sweep.policy == policy) &
                                       np.isclose(pair_sweep.left_threshold,
                                                  pair_sweep.right_threshold)].sort_values('left_threshold')
    assert len(equal_threshold_curve) == len(PAIR_THRESHOLDS)
    fig, axis = plt.subplots(figsize=(8, 4.5))
    left_line, = axis.plot(equal_threshold_curve.left_threshold,
                           equal_threshold_curve.left_recall * 100,
                           marker='o', color='tab:blue', label='Left recall')
    right_line, = axis.plot(equal_threshold_curve.left_threshold,
                            equal_threshold_curve.right_recall * 100,
                            marker='o', color='tab:green', label='Right recall')
    axis.set(xlabel='Equal left/right threshold', ylabel='Recall (%)', ylim=(0, 100),
             title='Validation recall and all-original normal-use FP/min')
    fp_axis = axis.twinx()
    fp_line, = fp_axis.plot(equal_threshold_curve.left_threshold,
                            equal_threshold_curve.normal_usage_fp_per_min,
                            marker='s', color='tab:red', label='Normal-use false positives / min')
    fp_axis.set_ylabel('False positives / min', color='tab:red')
    fp_axis.tick_params(axis='y', labelcolor='tab:red')
    axis.legend([left_line, right_line, fp_line],
                [line.get_label() for line in (left_line, right_line, fp_line)],
                loc='best', fontsize=8)
    fig.suptitle(
        f'{policy}: {len(normal_usage_recordings)} full normal_usage originals '
        f'({normal_usage_manifest.minutes.sum():.2f} min), including sources with training crops',
        y=1.02,
    )
    fig.tight_layout()
    plt.show()


In [ ]:
pair_sweep[(pair_sweep['left_threshold']  == 0.8) & (pair_sweep['right_threshold'] == 0.8)]

In [ ]:
def select_operating_point(candidates, mode):
    candidates = candidates.copy()
    rate = FP_SELECTION_METRIC
    if mode.startswith('recall_target_'):
        candidates = candidates[(candidates.left_recall >= RECALL_TARGET) &
                                (candidates.right_recall >= RECALL_TARGET)]
    if mode in ('fp_budget_max_recall', 'timely_recall_fp_budget'):
        candidates = candidates[np.isfinite(candidates[rate]) & (candidates[rate] < FP_PER_MIN_LIMIT)]
    if mode == 'recall_target_min_fp':
        candidates = candidates[np.isfinite(candidates[rate])]
    if candidates.empty:
        return None
    candidates['mean_threshold'] = (candidates.left_threshold + candidates.right_threshold) / 2
    candidates['minus_fp_rate'] = -candidates[rate]
    if mode == 'max_f1':
        sort_by = ['macro_f1', 'minus_fp_rate', 'macro_recall', 'mean_threshold']
    elif mode == 'recall_target_max_threshold':
        sort_by = ['mean_threshold', 'minus_fp_rate', 'macro_f1']
    elif mode == 'recall_target_min_fp':
        sort_by = ['minus_fp_rate', 'macro_f1', 'mean_threshold']
    elif mode == 'fp_budget_max_recall':
        sort_by = ['macro_recall', 'macro_f1', 'minus_fp_rate', 'mean_threshold']
    elif mode == 'timely_recall_fp_budget':
        sort_by = ['macro_timely_recall', 'macro_f1', 'minus_fp_rate', 'mean_threshold']
    else:
        raise ValueError(mode)
    # Final, deterministic ties: higher left threshold, then higher right threshold.
    return candidates.sort_values(sort_by + ['left_threshold', 'right_threshold'],
                                  ascending=False, kind='stable').iloc[0]

selected_rows = []
for policy in POLICIES:
    candidates = pair_sweep[pair_sweep.policy == policy]
    for mode in SELECTION_MODES:
        winner = select_operating_point(candidates, mode)
        reason = ''
        if winner is None:
            if (mode in ('fp_budget_max_recall', 'timely_recall_fp_budget', 'recall_target_min_fp')
                    and not np.isfinite(candidates[FP_SELECTION_METRIC]).any()):
                reason = f'{FP_SELECTION_METRIC} is undefined: no eligible evaluation minutes'
            elif mode.startswith('recall_target_'):
                reason = 'No pair achieves the recall target on both sides'
            else:
                reason = f'No pair is strictly below the {FP_SELECTION_METRIC} limit'
        selected_rows.append(dict(selection_mode=mode, policy=policy,
                                  feasible=winner is not None, infeasible_reason=reason,
                                  left_threshold=winner.left_threshold if winner is not None else np.nan,
                                  right_threshold=winner.right_threshold if winner is not None else np.nan,
                                  macro_f1=winner.macro_f1 if winner is not None else np.nan,
                                  left_recall=winner.left_recall if winner is not None else np.nan,
                                  right_recall=winner.right_recall if winner is not None else np.nan,
                                  fp_per_min=winner.fp_per_min if winner is not None else np.nan,
                                  negative_fp_per_min=winner.negative_fp_per_min if winner is not None else np.nan,
                                  normal_usage_fp_per_min=winner.get('normal_usage_fp_per_min', np.nan) if winner is not None else np.nan,
                                  fp_selection_metric=FP_SELECTION_METRIC,
                                  macro_timely_recall=winner.macro_timely_recall if winner is not None else np.nan))
selected_thresholds = pd.DataFrame(selected_rows)
for row in selected_thresholds[selected_thresholds.feasible].itertuples():
    if row.selection_mode.startswith('recall_target_'):
        assert row.left_recall >= RECALL_TARGET and row.right_recall >= RECALL_TARGET
    if row.selection_mode in ('fp_budget_max_recall', 'timely_recall_fp_budget'):
        assert getattr(row, FP_SELECTION_METRIC) < FP_PER_MIN_LIMIT
selected_thresholds

## 8. Original single-threshold diagnostics
`default_summary` and `metrics_summary` preserve the earlier 0.5-threshold baseline, single-threshold sweep and training history. They are **not** the selected-pair comparison; Section 9 gives those results across all levels. Window, frame and event metrics use different units.

In [ ]:
summary_parts = []
def collect_metrics(table, level, fixed=None):
    fixed = fixed or {}
    keys = [key for key in ('policy', 'threshold', 'class_id', 'epoch', 'budget_frames') if key in table.columns]
    return table.melt(id_vars=keys, var_name='metric', value_name='value').assign(level=level, **fixed)

summary_parts.append(collect_metrics(window_summary, 'window', {'threshold': cfg.training.prediction_threshold}))
summary_parts.append(collect_metrics(per_class, 'window_class'))
summary_parts.append(collect_metrics(frame_summary, 'frame', {'threshold': cfg.training.prediction_threshold}))
summary_parts.append(collect_metrics(event_summary, 'event'))
summary_parts.append(collect_metrics(event_per_class, 'event_class'))
summary_parts.append(collect_metrics(latency_summary, 'latency'))
summary_parts.append(collect_metrics(deadline_curve, 'deadline_curve'))
summary_parts.append(collect_metrics(pd.DataFrame([{'epoch': checkpoint['epoch'], **saved_metrics}]),
                                     'saved_checkpoint', {'threshold': cfg.training.prediction_threshold}))
cm_counts = pd.DataFrame([{'metric': f'window_cm_true_{t}_pred_{p}',
                           'value': int(window_cm.loc[t, p])} for t in range(3) for p in range(3)])
summary_parts.append(cm_counts.assign(level='window_confusion'))
if not history.empty:
    summary_parts.append(collect_metrics(history, 'epoch_history'))
metrics_summary = pd.concat(summary_parts, ignore_index=True)
metrics_summary = metrics_summary[['level', 'policy', 'threshold', 'class_id', 'epoch', 'budget_frames', 'metric', 'value']]
default_summary = pd.DataFrame([
    dict(level='window', policy='independent windows', threshold=cfg.training.prediction_threshold,
         precision=window_summary.precision.iloc[0], recall=window_summary.recall.iloc[0],
         f1=window_summary.f1.iloc[0], pr_auc=window_summary.pr_auc.iloc[0],
         false_alarms_per_min=np.nan, recall_lt_5f=np.nan, recall_lt_10f=np.nan,
         recall_lt_20f=np.nan, mean_latency_ms=np.nan),
    dict(level='frame', policy='continuous causal probabilities', threshold=cfg.training.prediction_threshold,
         precision=frame_summary.precision.iloc[0], recall=frame_summary.recall.iloc[0],
         f1=frame_summary.f1.iloc[0], pr_auc=frame_summary.pr_auc.iloc[0],
         false_alarms_per_min=np.nan, recall_lt_5f=np.nan, recall_lt_10f=np.nan,
         recall_lt_20f=np.nan, mean_latency_ms=np.nan),
] + [dict(level='event', policy=row.policy, threshold=row.threshold, precision=row.precision,
          recall=row.recall, f1=row.f1, pr_auc=np.nan, false_alarms_per_min=row.negative_fp_per_min,
          **{f'recall_lt_{d}f': float(latency_summary.loc[(latency_summary.policy == row.policy) &
                    (latency_summary.threshold == row.threshold), f'recall_lt_{d}f'].iloc[0])
             for d in (5, 10, 20)},
          mean_latency_ms=float(latency_summary.loc[(latency_summary.policy == row.policy) &
                                    (latency_summary.threshold == row.threshold), 'mean_ms'].iloc[0]))
     for row in event_summary[event_summary.threshold == cfg.training.prediction_threshold].itertuples()])
default_summary

In [ ]:
metrics_summary  # Filter level, policy, threshold, class_id or metric to see all calculated numbers.

## 9. Compare selected thresholds at every level
Every selected pair is replayed on window, frame and continuous event predictions. All alarm-based event metrics and threshold selection use the configured −20/+30-frame matching interval, including per-side, per-recording, per-type, binary, and deadline metrics. Window classification evaluates the whole window; frame classification evaluates frame labels rather than alarm matches, so the event tolerance does not change those labels. Window/frame PR-AUC and ROC-AUC depend only on raw probabilities; threshold selection cannot improve them. Infeasible modes keep an explicit row with no fabricated metrics. These are tuned-validation, **not** held-out test scores.

In [ ]:
import hashlib
import json

validation_sources = {str(path.relative_to(ROOT)) for rec in recordings
                      for path in (rec.path, rec.path.with_suffix('.txt'))}
validation_sources.add(str(Path(evaluation_data['session_exclusions_file']).relative_to(ROOT)))
validation_manifest = sorted((item for item in data_manifest if item['path'] in validation_sources),
                             key=lambda item: item['path'])
assert len(validation_manifest) == 2 * len(recordings) + 1
validation_sha256 = hashlib.sha256(json.dumps(validation_manifest, sort_keys=True).encode()).hexdigest()
checkpoint_sha256 = hashlib.sha256((RUN_DIR / 'best.pt').read_bytes()).hexdigest()
preprocessing_sha256 = hashlib.sha256(json.dumps(
    {'data': evaluation_data, 'labels': cfg.to_dict()['labels'], 'seed': cfg.seed},
    sort_keys=True).encode()).hexdigest()
evaluation_context = dict(
    checkpoint_path=str((RUN_DIR / 'best.pt').relative_to(ROOT)),
    data=evaluation_data, labels=cfg.to_dict()['labels'], seed=cfg.seed,
    checkpoint_training_seed=checkpoint_training_seed,
    validation_sha256=validation_sha256, preprocessing_sha256=preprocessing_sha256,
    duration_basis=DURATION_BASIS, duration_sha256=duration_sha256,
    normal_usage=dict(source_dir=str(NORMAL_USAGE_SOURCE_DIR),
                      files=normal_usage_manifest.file.tolist(), sha256=normal_usage_sha256,
                      manifest=normal_usage_manifest.to_dict('records')),
)
comparison_metadata = dict(model_id=RUN_DIR.name, checkpoint_sha256=checkpoint_sha256,
                           checkpoint_epoch=checkpoint['epoch'],
                           checkpoint_training_seed=checkpoint_training_seed,
                           validation_sha256=validation_sha256,
                           preprocessing_sha256=preprocessing_sha256,
                           selection_split='valid', evaluation_split='valid',
                           evaluation_design='tuned_validation',
                           threshold_grid=','.join(map(str, PAIR_THRESHOLDS)),
                           recall_target=RECALL_TARGET, fp_per_min_limit=FP_PER_MIN_LIMIT,
                           fp_selection_metric=FP_SELECTION_METRIC,
                           duration_basis=DURATION_BASIS, duration_sha256=duration_sha256,
                           normal_usage_sha256=normal_usage_sha256,
                           timely_budget_frames=TIMELY_BUDGET_FRAMES,
                           early_allowance_frames=EARLY_ALLOWANCE_FRAMES,
                           max_late_frames=MAX_LATE_FRAMES, refractory_sec=.5,
                           frame_label_half_width=10)
pd.DataFrame([comparison_metadata])

In [ ]:
def classification_results(truth, pred, scores):
    binary = binary_metrics(truth > 0, pred > 0)
    binary |= ranking_metrics(truth > 0, scores.max(axis=1))
    cm = np.zeros((3, 3), dtype=int)
    np.add.at(cm, (truth, pred), 1)
    binary.update(n_samples=len(truth), class_accuracy=float((truth == pred).mean()),
                  **{f'cm_{i}{j}': int(cm[i, j]) for i in range(3) for j in range(3)})
    per_side = {}
    for cls in (1, 2):
        per_side[cls] = (binary_metrics(truth == cls, pred == cls) |
                         ranking_metrics(truth == cls, scores[:, cls - 1]) |
                         dict(n_samples=len(truth), support=int((truth == cls).sum())))
    binary['macro_f1'] = float(np.nanmean([per_side[c]['f1'] for c in (1, 2)]))
    return binary, per_side

comparison_rows = []
for selected in selected_thresholds.itertuples(index=False):
    identifiers = dict(selection_mode=selected.selection_mode, policy=selected.policy,
                       left_threshold=selected.left_threshold, right_threshold=selected.right_threshold,
                       feasible=selected.feasible, **comparison_metadata)
    if not selected.feasible:
        comparison_rows.append(dict(**identifiers, level='event', side='all'))
        continue
    tl, tr = float(selected.left_threshold), float(selected.right_threshold)
    window_pred = choose_classes(window_side_scores, tl, tr)
    frame_pred = choose_classes(frame_side_scores, tl, tr)
    event_all, event_sides = evaluate_pair_events(tl, tr, selected.policy)
    window_all, window_sides = classification_results(window_truth, window_pred, window_side_scores)
    frame_all, frame_sides = classification_results(frame_true_class, frame_pred, frame_side_scores)
    for level, overall, sides in [('window', window_all, window_sides),
                                  ('frame', frame_all, frame_sides),
                                  ('event', event_all, event_sides)]:
        for label, scores in [('all', overall), ('left', sides[1]), ('right', sides[2])]:
            comparison_rows.append(dict(**identifiers, level=level, side=label, **scores))
comparison_summary = pd.DataFrame(comparison_rows)
for selected in selected_thresholds.itertuples(index=False):
    if not selected.feasible:
        continue
    event_row = comparison_summary[(comparison_summary.selection_mode == selected.selection_mode) &
                                   (comparison_summary.policy == selected.policy) &
                                   (comparison_summary.level == 'event') &
                                   (comparison_summary.side == 'all')].iloc[0]
    sweep_row = pair_sweep[(pair_sweep.policy == selected.policy) &
                           (pair_sweep.left_threshold == selected.left_threshold) &
                           (pair_sweep.right_threshold == selected.right_threshold)].iloc[0]
    assert (event_row.tp, event_row.fp, event_row.fn) == (sweep_row.tp, sweep_row.fp, sweep_row.fn)
assert np.array_equal(choose_classes(window_side_scores, .5, .5), windows.pred.to_numpy())
assert np.array_equal(choose_classes(frame_side_scores, .5, .5) > 0,
                      frames.event_score.to_numpy() >= .5)
comparison_summary[(comparison_summary.level == 'event') & (comparison_summary.side == 'all')][
    ['selection_mode', 'policy', 'left_threshold', 'right_threshold', 'feasible',
     'precision', 'recall', 'f1', 'fp_per_min', 'negative_fp_per_min', 'normal_usage_fp_per_min',
     'recall_lt_5f', 'recall_lt_10f', 'mean_latency_ms']]

In [ ]:
comparison_summary  # Full wide table: every selection mode/policy, level, and side.

In [ ]:
identity = ['model_id', 'checkpoint_sha256', 'checkpoint_epoch', 'validation_sha256',
            'preprocessing_sha256', 'duration_basis', 'duration_sha256', 'normal_usage_sha256', 'selection_split',
            'evaluation_split', 'evaluation_design', 'threshold_grid', 'recall_target',
            'fp_per_min_limit', 'fp_selection_metric', 'timely_budget_frames', 'early_allowance_frames',
            'max_late_frames', 'refractory_sec', 'frame_label_half_width',
            'selection_mode', 'policy', 'left_threshold', 'right_threshold',
            'feasible', 'level', 'side']
comparison_metrics = comparison_summary.melt(id_vars=identity, var_name='metric', value_name='value')
comparison_metrics = comparison_metrics.dropna(subset=['value']).reset_index(drop=True)
comparison_metrics  # Long-format numeric metrics; filter or pivot for model comparisons.

## 10. Which recording types are difficult?
Choose **one feasible selection mode and policy** from `selected_thresholds`. The left/right thresholds are then applied unchanged to every validation recording. `recording_type` comes from the filename: left/right double taps are grouped into `knock_twice`, while full `negative_mining` recordings and mined `__crop_f…_f…` CSVs are grouped into `hard_negative`. `switch_doubletap` is a separate standard **negative** type: `imu_<collector>_switch_doubletap_<timestamp>.csv`, with `action_label=switch_doubletap` and 20 × 3-second sections (60 seconds). It uses normal collection timing, not strict tap cues or continuous negative mining. Its label `.txt` must be empty; every window is `none` and every emitted alarm is a false positive.

The confusion plots are **row-normalized window classifications** (true none/left/right vs predicted none/left/right), using the dataset's 3-second windows. A missing true class is N/A, not 0% accuracy. Continuous **event** TP/FP/FN and FP/min are in the adjacent tables; events have no meaningful true-negative count. All rates are calculated from pooled counts rather than averages of file-level percentages. Latency is omitted. A type represented by one file provides limited evidence.

In [ ]:
TYPE_SELECTION_MODE = 'recall_target_max_threshold'  # One of the modes in selected_thresholds.
TYPE_POLICY = 'lookahead_12f'  # Or 'instant'; shared refractory period is preserved.
chosen_rows = selected_thresholds[(selected_thresholds.selection_mode == TYPE_SELECTION_MODE) &
                                  (selected_thresholds.policy == TYPE_POLICY)]
assert len(chosen_rows) == 1, 'Choose an existing selection mode and policy.'
chosen = chosen_rows.iloc[0]
if not chosen.feasible:
    raise ValueError(f'{TYPE_SELECTION_MODE} / {TYPE_POLICY} is infeasible: {chosen.infeasible_reason}')
type_left_threshold, type_right_threshold = float(chosen.left_threshold), float(chosen.right_threshold)
chosen_rows

In [ ]:
type_from_filename = recording_type_from_filename

assert type_from_filename('imu_CC_arbitrary_20260901_154914.csv') == 'arbitrary'
assert type_from_filename('imu_CC_knock_twice_left_20260901_153013.csv') == 'knock_twice'
assert type_from_filename('imuStrict_Miguel_knock_twice_left_20260929_115836.csv') == 'knock_twice'
assert type_from_filename('imu_Miguel_negative_mining_20261009_164938.csv') == 'hard_negative'
assert type_from_filename('imu_Miguel_negative_mining_20261009_164938__crop_f000495_f000795.csv') == 'hard_negative'
assert type_from_filename('imu_Miguel_switch_doubletap_20261012_170000.csv') == 'switch_doubletap'
assert type_from_filename('imu_Miguel_normal_usage_20261014_120000.csv') == 'normal_usage'
assert type_from_filename('imu_Miguel_normal_usage_20261014_120000__crop_f000000_f000300.csv') == 'hard_negative'

selected_window_predictions = choose_classes(window_side_scores, type_left_threshold, type_right_threshold)
recording_rows = []
for rec in recordings:
    file = rec.path.name
    positions = np.flatnonzero(windows.file.to_numpy() == file)
    assert len(positions), f'No validation windows for {file}'
    actual = window_truth[positions]
    predicted = selected_window_predictions[positions]
    if type_from_filename(file) == 'switch_doubletap':
        assert rec.action == 'switch_doubletap', f'Unexpected action label for {file}'
        assert not rec.events and np.all(actual == 0), f'switch_doubletap must have only negative labels: {file}'
    cm = np.zeros((3, 3), dtype=int)
    np.add.at(cm, (actual, predicted), 1)
    window_tp = int(cm[1:, 1:].sum())  # Binary: wrong side is still a detected event.
    window_fp, window_fn, window_tn = int(cm[0, 1:].sum()), int(cm[1:, 0].sum()), int(cm[0, 0])
    detected = emit_alarms_pair(probabilities[file], type_left_threshold, type_right_threshold,
                                POLICIES[TYPE_POLICY], round(.5 * rec.fs))
    matched = match_alarms(rec.events, detected)  # Class-aware, configured early/late allowance.
    binary_matched = match_alarms(rec.events, detected, class_aware=False)
    event_tp, event_fp, event_fn = len(matched), len(detected)-len(matched), len(rec.events)-len(matched)
    duration_info = duration_lookup.loc[file]
    minutes = duration_info.minutes
    by_side = {}
    for cls, label in ((1, 'left'), (2, 'right')):
        ctp = sum(rec.events[i][1] == cls for i, _ in matched)
        cfp = sum(alarm['class_id'] == cls for alarm in detected) - ctp
        cfn = sum(event_cls == cls for _, event_cls in rec.events) - ctp
        by_side.update({f'event_{label}_{metric}': value for metric, value in
                        dict(tp=ctp, fp=cfp, fn=cfn).items()})
    recording_rows.append(dict(selection_mode=TYPE_SELECTION_MODE, policy=TYPE_POLICY,
                               left_threshold=type_left_threshold, right_threshold=type_right_threshold,
                               file=file, user=rec.user, action=rec.action,
                               recording_type=type_from_filename(file), minutes=minutes,
                               evaluated_minutes=duration_info.evaluated_minutes,
                               source_file=duration_info.source_file, source_sha256=duration_info.source_sha256,
                               source_minutes=duration_info.source_minutes, source_members=duration_info.source_members,
                               n_windows=len(positions), window_correct=int(np.trace(cm)),
                               window_tp=window_tp, window_fp=window_fp, window_fn=window_fn, window_tn=window_tn,
                               window_accuracy=divide(np.trace(cm), len(positions)),
                               window_precision=divide(window_tp, window_tp+window_fp),
                               window_recall=divide(window_tp, window_tp+window_fn),
                               window_f1=divide(2*window_tp, 2*window_tp+window_fp+window_fn),
                               window_false_positive_rate=divide(window_fp, window_fp+window_tn),
                               n_true_events=len(rec.events), n_alarms=len(detected),
                               event_tp=event_tp, event_fp=event_fp, event_fn=event_fn,
                               binary_event_tp=len(binary_matched),
                               event_precision=divide(event_tp, event_tp+event_fp),
                               event_recall=divide(event_tp, event_tp+event_fn),
                               event_f1=divide(2*event_tp, 2*event_tp+event_fp+event_fn),
                               event_fp_per_min=divide(event_fp, minutes),
                               **by_side,
                               **{f'cm_{i}{j}': int(cm[i, j]) for i in range(3) for j in range(3)}))
recording_metrics = pd.DataFrame(recording_rows).assign(**comparison_metadata)
assert recording_metrics.n_windows.sum() == len(windows)
overall_selected = comparison_summary[(comparison_summary.selection_mode == TYPE_SELECTION_MODE) &
                                      (comparison_summary.policy == TYPE_POLICY) &
                                      (comparison_summary.side == 'all')]
for level, prefix in (('window', 'window'), ('event', 'event')):
    total = overall_selected[overall_selected.level == level].iloc[0]
    assert all(recording_metrics[f'{prefix}_{name}'].sum() == total[name]
               for name in ('tp', 'fp', 'fn'))
assert all(recording_metrics[f'cm_{i}{j}'].sum() ==
           overall_selected[overall_selected.level == 'window'].iloc[0][f'cm_{i}{j}']
           for i in range(3) for j in range(3))
recording_metrics[['file', 'user', 'recording_type', 'n_windows', 'n_true_events',
                   'window_accuracy', 'window_fp', 'window_fn', 'event_tp', 'event_fp',
                   'event_fn', 'event_fp_per_min']].sort_values('event_fp_per_min', ascending=False)

In [ ]:
recording_metrics  # Full filterable table, including per-file 3x3 counts and provenance.

In [ ]:
type_rows, confusion_rows = [], []
for recording_type, group in recording_metrics.groupby('recording_type', sort=True):
    cm = np.array([[int(group[f'cm_{i}{j}'].sum()) for j in range(3)] for i in range(3)])
    wtp, wfp, wfn, wtn = (int(group[f'window_{key}'].sum()) for key in ('tp', 'fp', 'fn', 'tn'))
    etp, efp, efn = (int(group[f'event_{key}'].sum()) for key in ('tp', 'fp', 'fn'))
    minutes = group.minutes.sum()
    summary = dict(selection_mode=TYPE_SELECTION_MODE, policy=TYPE_POLICY,
                   left_threshold=type_left_threshold, right_threshold=type_right_threshold,
                   recording_type=recording_type, recordings=len(group), minutes=minutes,
                   evaluated_minutes=group.evaluated_minutes.sum(),
                   n_windows=group.n_windows.sum(), n_true_events=group.n_true_events.sum(),
                   **{f'n_windows_{label}': int(cm[i].sum())
                      for i, label in enumerate(('none', 'left', 'right'))},
                   window_tp=wtp, window_fp=wfp, window_fn=wfn, window_tn=wtn,
                   window_accuracy=divide(np.trace(cm), cm.sum()),
                   window_precision=divide(wtp, wtp+wfp), window_recall=divide(wtp, wtp+wfn),
                   window_f1=divide(2*wtp, 2*wtp+wfp+wfn),
                   window_false_positive_rate=divide(wfp, wfp+wtn),
                   event_tp=etp, event_fp=efp, event_fn=efn,
                   event_precision=divide(etp, etp+efp), event_recall=divide(etp, etp+efn),
                   event_f1=divide(2*etp, 2*etp+efp+efn), event_fp_per_min=divide(efp, minutes),
                   **{f'event_{side}_{key}': int(group[f'event_{side}_{key}'].sum())
                      for side in ('left', 'right') for key in ('tp', 'fp', 'fn')})
    for side in ('left', 'right'):
        tp, fp, fn = (summary[f'event_{side}_{key}'] for key in ('tp', 'fp', 'fn'))
        summary[f'event_{side}_precision'] = divide(tp, tp+fp)
        summary[f'event_{side}_recall'] = divide(tp, tp+fn)
        summary[f'event_{side}_f1'] = divide(2*tp, 2*tp+fp+fn)
        cls = 1 if side == 'left' else 2
        correct = int(cm[cls, cls])
        summary[f'window_{side}_precision'] = divide(correct, cm[:, cls].sum())
        summary[f'window_{side}_recall'] = divide(correct, cm[cls].sum())
    type_rows.append(summary)
    for i, actual in enumerate(('none', 'left', 'right')):
        n = int(cm[i].sum())
        for j, predicted in enumerate(('none', 'left', 'right')):
            confusion_rows.append(dict(selection_mode=TYPE_SELECTION_MODE, policy=TYPE_POLICY,
                                       recording_type=recording_type, true_class=actual,
                                       predicted_class=predicted, count=int(cm[i, j]),
                                       true_count=n, row_fraction=divide(cm[i, j], n)))
type_metrics = pd.DataFrame(type_rows).assign(**comparison_metadata)
type_confusion = pd.DataFrame(confusion_rows).assign(**comparison_metadata)
assert type_metrics.n_windows.sum() == len(windows)
assert type_metrics.event_fp.sum() == recording_metrics.event_fp.sum()
type_metrics[['recording_type', 'recordings', 'n_windows', 'n_true_events',
              'window_accuracy', 'window_fp', 'window_fn', 'window_false_positive_rate',
              'event_tp', 'event_fp', 'event_fn', 'event_fp_per_min',
              'event_left_recall', 'event_right_recall']]

In [ ]:
import math
types = sorted(type_metrics.recording_type)
ncols = 2
fig, axes = plt.subplots(math.ceil(len(types) / ncols), ncols,
                         figsize=(13, 3.8 * math.ceil(len(types) / ncols)), squeeze=False)
for ax, recording_type in zip(axes.flat, types):
    group = type_confusion[type_confusion.recording_type == recording_type]
    counts = group['count'].to_numpy().reshape(3, 3)
    totals = counts.sum(axis=1)
    rates = np.divide(counts, totals[:, None], out=np.full((3, 3), np.nan),
                      where=totals[:, None] > 0)
    colors = plt.get_cmap('Blues').copy()
    colors.set_bad('#eeeeee')
    image = ax.imshow(np.ma.masked_invalid(rates), vmin=0, vmax=1, cmap=colors)
    ax.set(title=f'{recording_type} ({int(counts.sum())} windows)',
           xlabel='Predicted', ylabel='Actual', xticks=range(3), yticks=range(3),
           xticklabels=('none', 'left', 'right'),
           yticklabels=[f'{label} (n={n})' for label, n in zip(('none','left','right'), totals)])
    for i in range(3):
        for j in range(3):
            text = f'{rates[i, j]:.0%}\n({counts[i, j]})' if totals[i] else 'N/A'
            ax.text(j, i, text, ha='center', va='center',
                    color='white' if totals[i] and rates[i, j] > .55 else 'black')
for ax in list(axes.flat)[len(types):]:
    ax.axis('off')
fig.colorbar(image, ax=list(axes.flat), fraction=.02, pad=.03, label='Fraction of true row')
fig.suptitle(f'Window confusion by recording type — {TYPE_SELECTION_MODE} / {TYPE_POLICY}', y=1.01)
plt.show()

In [ ]:
type_confusion[['recording_type', 'true_class', 'predicted_class', 'count', 'true_count', 'row_fraction']]  # Full table: type_confusion.

In [ ]:
negative_types = type_metrics[type_metrics.n_true_events == 0].sort_values('event_fp_per_min', ascending=False)
if len(negative_types):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    labels = [f'{row.recording_type} (n={row.recordings})' for row in negative_types.itertuples()]
    axes[0].barh(labels, negative_types.event_fp_per_min)
    axes[0].set(xlabel='Unmatched alarms / recorded minute', title='Negative types: event false alarms')
    axes[1].barh(labels, 100 * negative_types.window_false_positive_rate)
    axes[1].set(xlabel='Positive predictions / negative windows (%)',
                title='Negative types: false-positive windows')
    for ax in axes: ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

In [ ]:
type_metrics  # Filter recording_type and compare window errors with event false alarms/minute.

### Per-recording and per-type exports for every feasible selection mode
The plots above use the chosen `TYPE_SELECTION_MODE` and `TYPE_POLICY`. For cross-model comparisons, the following tables additionally score each *feasible* selected pair, so the comparison notebook can switch modes without silently reusing the wrong type statistics. No threshold grid is rerun here.

In [ ]:
other_recordings = []
for selected in selected_thresholds[selected_thresholds.feasible].itertuples(index=False):
    if selected.selection_mode == TYPE_SELECTION_MODE and selected.policy == TYPE_POLICY:
        continue  # Reuse the detailed recording_metrics computed above.
    tl, tr = float(selected.left_threshold), float(selected.right_threshold)
    predicted_windows = choose_classes(window_side_scores, tl, tr)
    for rec in recordings:
        file = rec.path.name
        positions = np.flatnonzero(windows.file.to_numpy() == file)
        cm = np.zeros((3, 3), dtype=int)
        np.add.at(cm, (window_truth[positions], predicted_windows[positions]), 1)
        alarms_for_file = emit_alarms_pair(probabilities[file], tl, tr, POLICIES[selected.policy],
                                          round(.5 * rec.fs))
        matches = match_alarms(rec.events, alarms_for_file)
        side_counts = {}
        for cls, label in ((1, 'left'), (2, 'right')):
            tp = sum(rec.events[i][1] == cls for i, _ in matches)
            side_counts.update({f'event_{label}_tp': tp,
                                f'event_{label}_fp': sum(a['class_id'] == cls for a in alarms_for_file) - tp,
                                f'event_{label}_fn': sum(c == cls for _, c in rec.events) - tp})
        other_recordings.append(dict(selection_mode=selected.selection_mode, policy=selected.policy,
                                     left_threshold=tl, right_threshold=tr, file=file, user=rec.user,
                                     recording_type=type_from_filename(file), minutes=duration_lookup.loc[file, 'minutes'],
                                     evaluated_minutes=duration_lookup.loc[file, 'evaluated_minutes'],
                                     source_file=duration_lookup.loc[file, 'source_file'],
                                     source_sha256=duration_lookup.loc[file, 'source_sha256'],
                                     source_minutes=duration_lookup.loc[file, 'source_minutes'],
                                     source_members=duration_lookup.loc[file, 'source_members'],
                                     n_windows=len(positions), n_true_events=len(rec.events),
                                     window_tp=int(cm[1:, 1:].sum()), window_fp=int(cm[0, 1:].sum()),
                                     window_fn=int(cm[1:, 0].sum()), window_tn=int(cm[0, 0]),
                                     event_tp=len(matches), event_fp=len(alarms_for_file)-len(matches),
                                     event_fn=len(rec.events)-len(matches), **side_counts,
                                     **{f'cm_{i}{j}': int(cm[i, j]) for i in range(3) for j in range(3)}))
recording_metrics_all_modes = pd.concat(
    [recording_metrics, pd.DataFrame(other_recordings)], ignore_index=True).assign(**comparison_metadata)
recording_metrics_all_modes['event_fp_per_min'] = (recording_metrics_all_modes.event_fp /
                                                    recording_metrics_all_modes.minutes)
recording_metrics_all_modes['window_accuracy'] = (
    sum(recording_metrics_all_modes[f'cm_{i}{i}'] for i in range(3)) /
    recording_metrics_all_modes.n_windows)
assert len(recording_metrics_all_modes) == len(recordings) * int(selected_thresholds.feasible.sum())
recording_metrics_all_modes[['selection_mode', 'policy', 'recording_type', 'file',
                             'event_tp', 'event_fp', 'event_fn', 'window_fp', 'window_fn']]

In [ ]:
all_type_rows = []
for (mode, policy, tl, tr, recording_type), group in recording_metrics_all_modes.groupby(
        ['selection_mode', 'policy', 'left_threshold', 'right_threshold', 'recording_type'], sort=True):
    cm = np.array([[int(group[f'cm_{i}{j}'].sum()) for j in range(3)] for i in range(3)])
    wtp, wfp, wfn, wtn = (int(group[f'window_{name}'].sum()) for name in ('tp', 'fp', 'fn', 'tn'))
    etp, efp, efn = (int(group[f'event_{name}'].sum()) for name in ('tp', 'fp', 'fn'))
    minutes = group.minutes.sum()
    row = dict(selection_mode=mode, policy=policy, left_threshold=tl, right_threshold=tr,
               recording_type=recording_type, recordings=len(group), minutes=minutes,
               evaluated_minutes=group.evaluated_minutes.sum(),
               n_windows=int(cm.sum()), n_true_events=int(group.n_true_events.sum()),
               window_tp=wtp, window_fp=wfp, window_fn=wfn, window_tn=wtn,
               window_accuracy=divide(np.trace(cm), cm.sum()),
               window_precision=divide(wtp, wtp+wfp), window_recall=divide(wtp, wtp+wfn),
               window_f1=divide(2*wtp, 2*wtp+wfp+wfn),
               window_false_positive_rate=divide(wfp, wfp+wtn),
               event_tp=etp, event_fp=efp, event_fn=efn,
               event_precision=divide(etp, etp+efp), event_recall=divide(etp, etp+efn),
               event_f1=divide(2*etp, 2*etp+efp+efn), event_fp_per_min=divide(efp, minutes),
               **{f'cm_{i}{j}': int(cm[i, j]) for i in range(3) for j in range(3)})
    for side in ('left', 'right'):
        tp, fp, fn = (int(group[f'event_{side}_{name}'].sum()) for name in ('tp', 'fp', 'fn'))
        row.update({f'event_{side}_tp': tp, f'event_{side}_fp': fp,
                    f'event_{side}_fn': fn, f'event_{side}_recall': divide(tp, tp+fn),
                    f'event_{side}_precision': divide(tp, tp+fp),
                    f'event_{side}_f1': divide(2*tp, 2*tp+fp+fn)})
    all_type_rows.append(row)
type_metrics_all_modes = pd.DataFrame(all_type_rows).assign(**comparison_metadata)
for selected in selected_thresholds[selected_thresholds.feasible].itertuples(index=False):
    files = recording_metrics_all_modes[(recording_metrics_all_modes.selection_mode == selected.selection_mode) &
                                       (recording_metrics_all_modes.policy == selected.policy)]
    by_type = type_metrics_all_modes[(type_metrics_all_modes.selection_mode == selected.selection_mode) &
                                     (type_metrics_all_modes.policy == selected.policy)]
    overall = comparison_summary[(comparison_summary.selection_mode == selected.selection_mode) &
                                 (comparison_summary.policy == selected.policy) &
                                 (comparison_summary.side == 'all')]
    for prefix, level in (('window', 'window'), ('event', 'event')):
        expected = overall[overall.level == level].iloc[0]
        for name in ('tp', 'fp', 'fn'):
            assert files[f'{prefix}_{name}'].sum() == by_type[f'{prefix}_{name}'].sum() == expected[name]
type_metrics_all_modes[['selection_mode', 'policy', 'left_threshold', 'right_threshold',
                        'recording_type', 'recordings', 'event_tp', 'event_fp', 'event_fn',
                        'event_fp_per_min', 'window_false_positive_rate']]

### Optional CSV export
Set `SAVE_COMPARISON=True` to save the selected pairs, comparison, full sweep, and per-file/type tables (including all feasible modes). With an evaluation-data override, the default export directory is separate from the checkpoint so historical exports are preserved. Existing filenames in the chosen export folder will be overwritten if you rerun this cell.

In [ ]:
# Dedicated measured normal-use scores. Never add these counts to crop-based summaries.
normal_usage_file_parts, normal_usage_alarm_parts = [], []
for selected in selected_thresholds[selected_thresholds.feasible].itertuples(index=False):
    _, _, by_file, by_alarm = evaluate_normal_usage(
        normal_usage_recordings, normal_usage_probabilities, float(selected.left_threshold),
        float(selected.right_threshold), POLICIES[selected.policy])
    identifiers = dict(selection_mode=selected.selection_mode, policy=selected.policy,
                       left_threshold=selected.left_threshold, right_threshold=selected.right_threshold,
                       **comparison_metadata)
    normal_usage_file_parts.append(by_file.assign(**identifiers))
    normal_usage_alarm_parts.append(by_alarm.assign(**identifiers))
normal_usage_recording_metrics = pd.concat(normal_usage_file_parts, ignore_index=True)
normal_usage_alarm_details = pd.concat(normal_usage_alarm_parts, ignore_index=True)
normal_usage_metrics = comparison_summary[(comparison_summary.level == 'event') &
                                          (comparison_summary.side == 'all')][
    ['selection_mode', 'policy', 'left_threshold', 'right_threshold', 'feasible',
     'normal_usage_recordings', 'normal_usage_minutes', 'normal_usage_fp',
     'normal_usage_fp_per_min', 'normal_usage_sha256']].copy().assign(**comparison_metadata)
display(normal_usage_metrics)
display(normal_usage_recording_metrics)
SAVE_COMPARISON = False  # Opt in; exports go only to fast_evaluation/.
if SAVE_COMPARISON:
    export_dir = EVALUATION_OUTPUT_DIR / 'threshold_comparison'
    export_dir.mkdir(parents=True, exist_ok=True)
    for name, table in [('selected_thresholds', selected_thresholds),
                        ('comparison_summary', comparison_summary),
                        ('comparison_metrics', comparison_metrics),
                        ('pair_sweep', pair_sweep),
                        (f'recording_metrics_{TYPE_SELECTION_MODE}_{TYPE_POLICY}', recording_metrics),
                        (f'type_metrics_{TYPE_SELECTION_MODE}_{TYPE_POLICY}', type_metrics),
                        (f'type_confusion_{TYPE_SELECTION_MODE}_{TYPE_POLICY}', type_confusion),
                        ('recording_metrics_all_modes', recording_metrics_all_modes),
                        ('type_metrics_all_modes', type_metrics_all_modes),
                        ('normal_usage_metrics', normal_usage_metrics),
                        ('normal_usage_recording_metrics', normal_usage_recording_metrics),
                        ('normal_usage_alarm_details', normal_usage_alarm_details)]:
        exported = table.assign(**comparison_metadata) if name in ('selected_thresholds', 'pair_sweep') else table
        exported.to_csv(export_dir / f'{name}.csv', index=False)
    (export_dir / 'evaluation_context.json').write_text(
        json.dumps(evaluation_context, indent=2, sort_keys=True) + '\n')
    print(f'Saved comparison tables to {export_dir}')
    duration_audit.to_csv(export_dir / 'recording_duration_audit.csv', index=False)
    normal_usage_manifest.to_csv(export_dir / 'normal_usage_manifest.csv', index=False)
else:
    print('Set SAVE_COMPARISON = True to export the comparison and per-type CSV tables.')

In [ ]:
comparison_metrics